# Stage 2: clean the data

**Reads:** `data/gravity_extract.csv`, saved by stage 1.

**Writes:**
- `data/gravity_clean.csv`: the table every model uses. One row is one ordered pair of countries in one year with a recorded trade flow: the flow in thousand current USD, in levels, and the eight predictors.
- `results/cleaning_log.csv`: for each step and year, the rows before, dropped and left. Stage 7 turns it into the cleaning table of the paper.

The steps are those of section 6, stage 2 of `docs/build_plan.md`. Every count printed below is checked against section 3 of the plan, which Claude read on Yu's copy of the CEPII file on 3 October 2026. If a count differs, the run stops and prints both numbers.

*Drafted with Claude (Anthropic), which was asked for the stage that cleans the extract as the build plan sets out, checks every count against the plan, and gives a first look at the cleaned data. It replaces the placeholder of 3 October 2026.*

In [ ]:
from helpers import ANALYSIS_YEAR, DATA, DEV_YEAR, INDICATORS, MIN_FLOW, PREDICTORS, RESULTS, ROOT, TARGET, refuse_part

import hashlib

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

refuse_part()

EXTRACT = DATA / "gravity_extract.csv"
CLEAN = DATA / "gravity_clean.csv"
LOG = RESULTS / "cleaning_log.csv"

YEARS = [DEV_YEAR, ANALYSIS_YEAR]
KEY = ["year", "country_id_o", "country_id_d"]
# Country identifiers are text: "DEU.1" and "DEU.2" must not be read as numbers.
ID_COLUMNS = ["country_id_o", "country_id_d", "iso3_o", "iso3_d"]
SAVED_COLUMNS = ["year", "country_id_o", "country_id_d", "pair_id", TARGET] + PREDICTORS

# Section 3 of the build plan: the rows each step drops, per year.
EXPECTED_DROPPED = {
    1: {2017: 252, 2019: 252},
    2: {2017: 8_262, 2019: 8_262},
    3: {2017: 23_106, 2019: 23_870},
    4: {2017: 2_368, 2019: 2_551},
    5: {2017: 554, 2019: 462},
}
EXPECTED_USABLE = {2017: 28_962, 2019: 28_107}
# The counts of section 3 hold for 2017 and 2019 (decision 1), without a floor on small flows (decision 3).
CHECK_COUNTS = MIN_FLOW is None and YEARS == [2017, 2019]


def expect(name, year, value, expected):
    """Stop if a value differs from section 3 of the build plan, printing both."""
    if CHECK_COUNTS:
        assert value == expected, f"{name}, {year}: {value} here, {expected} in section 3 of the build plan. Stop and report both."


## Read the extract

Every empty field of the extract is a missing value. No other text, such as "NA", is read as missing.

In [ ]:
extract = pd.read_csv(EXTRACT, dtype={column: str for column in ID_COLUMNS}, keep_default_na=False, na_values=[""])
assert sorted(extract["year"].unique()) == YEARS, f"Years in the extract: {sorted(extract['year'].unique())}"
print(f"{EXTRACT.relative_to(ROOT)}: {len(extract):,} rows, {extract.shape[1]} columns")
print(extract.groupby("year").size().rename("rows").to_string())

Each step below drops some rows. The function `drop` prints, for each year, the rows before the step, the rows it drops and the rows left, adds the same three numbers to the cleaning log, and checks the rows dropped against section 3 of the plan.

In [ ]:
log = []


def drop(table, remove, step, reason):
    """Drop the rows where remove is True: print and log the counts per year, check them, return the rest."""
    print(f"Step {step}: drop rows where {reason}")
    for year in YEARS:
        in_year = table["year"] == year
        before = int(in_year.sum())
        dropped = int((in_year & remove).sum())
        left = before - dropped
        print(f"  {year}: {before:,} rows, {dropped:,} dropped, {left:,} left")
        log.append({"year": year, "step": step, "dropped_because": reason,
                    "rows_before": before, "rows_dropped": dropped, "rows_left": left})
        if step in EXPECTED_DROPPED:
            expect(f"Rows dropped at step {step}", year, dropped, EXPECTED_DROPPED[step][year])
    return table[~remove].copy()

## Step 1. Same origin and destination

The file pairs every country with itself as well, with no trade flow.

In [ ]:
table = drop(extract, extract["country_id_o"] == extract["country_id_d"], 1, "origin and destination are the same country")

## Step 2. Countries that do not exist in the year

The file lists every country identifier in every year, including countries that did not exist then, such as West Germany (`DEU.1`) in 2017.

In [ ]:
table = drop(table, (table["country_exists_o"] != 1) | (table["country_exists_d"] != 1), 2, "either country does not exist in that year")
print("Left: pairs of distinct countries that exist in the year.")

## Step 3. No recorded flow

`tradeflow_baci` never holds a zero: a pair with no recorded trade is missing. The models predict recorded flows only, so these rows go. The total of the recorded flows kept here is the "recorded trade" that the shares below refer to.

In [ ]:
table = drop(table, table[TARGET].isna(), 3, "no trade flow is recorded")

not_positive = int((table[TARGET] <= 0).sum())
print(f"Flows that are zero or negative: {not_positive} (expected: none)")
assert not_positive == 0, "A recorded flow is zero or negative: the Poisson deviance needs positive flows. Stop and report."

if MIN_FLOW is not None:
    # Decision 3: a floor on very small flows, in thousand USD. The counts of section 3 no longer apply.
    table = drop(table, table[TARGET] < MIN_FLOW, "3b", f"the flow is below MIN_FLOW = {MIN_FLOW}")

recorded_trade = table.groupby("year")[TARGET].sum()

## Step 4. GDP or distance missing

The GDP of a few countries is missing in the year, among them North Korea, Venezuela and, in 2019, Syria. Distance is never missing.

In [ ]:
for column in ["gdp_o", "gdp_d", "distw_harmonic"]:
    print(f"{column} missing:", table.groupby("year")[column].apply(lambda values: int(values.isna().sum())).to_dict())

remove = table["gdp_o"].isna() | table["gdp_d"].isna() | table["distw_harmonic"].isna()
for year in YEARS:
    in_year = table["year"] == year
    countries = sorted(set(table.loc[in_year & table["gdp_o"].isna(), "country_id_o"])
                       | set(table.loc[in_year & table["gdp_d"].isna(), "country_id_d"]))
    share = table.loc[in_year & remove, TARGET].sum() / recorded_trade[year]
    print(f"{year}: GDP missing for {len(countries)} countries: {', '.join(countries)}")
    print(f"{year}: these rows hold {100 * share:.2f}% of the recorded trade")
    expect("Countries with GDP missing", year, len(countries), {2017: 21, 2019: 23}[year])
    expect("Share of the recorded trade, in %", year, round(100 * share, 2), {2017: 0.34, 2019: 0.26}[year])
    expect("Rows with distance missing", year, int(table.loc[in_year, "distw_harmonic"].isna().sum()), 0)

table = drop(table, remove, 4, "the GDP of either country or the distance is missing")

## Step 5. An indicator missing (decision 10)

Two groups of rows, which do not overlap:
- `comlang_off` and `comcol` are missing for every row of a few territories.
- `col_dep_ever` is missing for six rows: China with Hong Kong, China with Taiwan, and Israel with Palestine, in both directions. The CEPII documentation (section 5.3) leaves these ties missing on purpose. Decision 10 of the plan drops them like every other row with a missing predictor. They carry a large share of the recorded trade, which is printed here.

In [ ]:
print("Missing, per indicator:")
print(table.groupby("year")[INDICATORS].apply(lambda values: values.isna().sum()).to_string())

for column in INDICATORS:
    missing = table[column].isna()
    # The countries concerned: those for which every remaining row has this indicator missing.
    both_ends = pd.concat([pd.DataFrame({"year": table["year"], "country": table["country_id_o"], "missing": missing}),
                           pd.DataFrame({"year": table["year"], "country": table["country_id_d"], "missing": missing})])
    per_country = both_ends.groupby(["year", "country"])["missing"].mean()
    always = per_country[per_country == 1]
    for year in YEARS:
        if missing[table["year"] == year].any():
            names = list(always.loc[year].index) if year in always.index.get_level_values(0) else []
            print(f"{year}: {column} missing in every row of: {', '.join(names) if names else 'no country'}")

language = table["comlang_off"].isna() | table["comcol"].isna()
colonial = table["col_dep_ever"].isna()
assert not (language & colonial).any(), "The two groups of missing indicators overlap."
for year in YEARS:
    in_year = table["year"] == year
    expect("Rows with comlang_off or comcol missing", year, int((in_year & language).sum()), {2017: 548, 2019: 456}[year])
    expect("Rows with col_dep_ever missing", year, int((in_year & colonial).sum()), 6)
    expect("Rows with contig or fta_wto missing", year, int(table.loc[in_year, ["contig", "fta_wto"]].isna().sum().sum()), 0)

print("\nThe rows with col_dep_ever missing, and their rank among the flows of the year (1 = largest):")
rank = table.groupby("year")[TARGET].rank(ascending=False, method="min").astype(int)
six = table.loc[colonial, ["year", "country_id_o", "country_id_d", TARGET]].assign(rank=rank[colonial])
print(six.to_string(index=False))
for year in YEARS:
    share = six.loc[six["year"] == year, TARGET].sum() / recorded_trade[year]
    print(f"{year}: these six rows hold {100 * share:.2f}% of the recorded trade")
    expect("Share of the recorded trade, in %", year, round(100 * share, 2), {2017: 2.47, 2019: 2.43}[year])
    china_to_hong_kong = six.loc[(six["year"] == year) & (six["country_id_o"] == "CHN") & (six["country_id_d"] == "HKG"), "rank"]
    expect("Rank of the flow from China to Hong Kong", year, int(china_to_hong_kong.iloc[0]), 4)

table = drop(table, table[INDICATORS].isna().any(axis=1), 5, "any of the five indicators is missing")
for year in YEARS:
    expect("Usable rows", year, int((table["year"] == year).sum()), EXPECTED_USABLE[year])

## Step 6. Indicators hold only 0 and 1

They are stored as integers.

In [ ]:
for column in INDICATORS:
    values = set(table[column].unique())
    assert values <= {0, 1}, f"{column} holds {values}"
    table[column] = table[column].astype(int)
print("Every indicator holds only 0 and 1.")

## Step 7. Natural logs of the GDPs and the distance

The gravity equation is multiplicative, so these three predictors enter in logs. The flow stays in levels: every model predicts the flow itself, in thousand USD (section 1 of the plan).

In [ ]:
table["ln_gdp_o"] = np.log(table["gdp_o"])
table["ln_gdp_d"] = np.log(table["gdp_d"])
table["ln_dist"] = np.log(table["distw_harmonic"])
for column in ["ln_gdp_o", "ln_gdp_d", "ln_dist"]:
    assert np.isfinite(table[column]).all(), f"{column} is not finite in every row"
print("ln_gdp_o, ln_gdp_d and ln_dist are finite in every row.")

## Step 8. No key appears twice

In [ ]:
assert not table.duplicated(KEY).any(), "Some (year, origin, destination) appears twice"
print("No (year, origin, destination) appears twice.")

## Step 9. The pair identifier

`pair_id` names the unordered pair: the two identifiers in alphabetical order, joined with `|`. Stage 3 splits the data by pair, so that the two directions of a pair are always on the same side of the split.

In [ ]:
origin, destination = table["country_id_o"], table["country_id_d"]
table["pair_id"] = np.where(origin < destination, origin + "|" + destination, destination + "|" + origin)

rows_per_pair = table.groupby(["year", "pair_id"]).size()
for year in YEARS:
    pairs = rows_per_pair.loc[year]
    print(f"{year}: {len(pairs):,} unordered pairs, {int((pairs == 2).sum()):,} with both directions")
    expect("Unordered pairs", year, len(pairs), {2017: 16_150, 2019: 15_620}[year])
    expect("Pairs with both directions", year, int((pairs == 2).sum()), {2017: 12_812, 2019: 12_487}[year])

## Step 10. Save the cleaned table and the cleaning log

The cleaned table keeps 13 columns: the key, the pair identifier, the flow and the eight predictors. Its SHA-256 fingerprint is printed, so that a run on another machine can be compared with this one: the same extract gives the same file.

In [ ]:
clean = table[SAVED_COLUMNS]
clean.to_csv(CLEAN, index=False, lineterminator="\n")
with open(CLEAN, "rb") as file:
    clean_sha256 = hashlib.file_digest(file, "sha256").hexdigest()
print(f"Saved {CLEAN.relative_to(ROOT)}: {len(clean):,} rows, {clean.shape[1]} columns, {CLEAN.stat().st_size / 1e6:.1f} MB")
print("SHA-256:", clean_sha256)

RESULTS.mkdir(parents=True, exist_ok=True)
cleaning_log = pd.DataFrame(log)
cleaning_log.to_csv(LOG, index=False, lineterminator="\n")
print(f"Saved {LOG.relative_to(ROOT)}")
print(cleaning_log.to_string(index=False))

## Step 11. A first look

Shown here and not saved. The log of the flow is for display only: no model uses it.

In [ ]:
print("First rows:")
print(clean.head(8).to_string(index=False))

In [ ]:
print("Summary statistics, per year:")
for year in YEARS:
    print(f"\n{year}")
    print(clean.loc[clean["year"] == year, [TARGET] + PREDICTORS].describe().T.to_string(float_format=lambda value: f"{value:,.3f}"))

In [ ]:
print("Share of rows where each indicator is 1:")
print(clean.groupby("year")[INDICATORS].mean().map(lambda share: f"{100 * share:.1f}%").to_string())

print("\nMissing values per column of the extract, per year:")
print(extract.groupby("year").apply(lambda rows: rows.isna().sum(), include_groups=False).T.to_string())

In [ ]:
for year in YEARS:
    flows = clean.loc[clean["year"] == year, TARGET].sort_values(ascending=False)
    top_1 = flows.iloc[: round(0.01 * len(flows))].sum() / flows.sum()
    top_10 = flows.iloc[: round(0.10 * len(flows))].sum() / flows.sum()
    below_one = int((flows < 1).sum())
    ln_flow = np.log(flows)
    print(f"{year}: the largest 1% of rows hold {100 * top_1:.1f}% of the total flow, the largest 10% hold {100 * top_10:.1f}%")
    print(f"{year}: {below_one:,} rows have a flow below 1 (under 1,000 USD)")
    print(f"{year}: log of the flow: mean {ln_flow.mean():.3f}, standard deviation {ln_flow.std():.3f}, minimum {ln_flow.min():.3f}")
    expect("Rows with a flow below 1", year, below_one, {2017: 1_562, 2019: 1_524}[year])
    expect("Mean of the log of the flow", year, round(ln_flow.mean(), 3), {2017: 7.535, 2019: 7.626}[year])
    expect("Standard deviation of the log of the flow", year, round(ln_flow.std(), 3), {2017: 4.387, 2019: 4.410}[year])
    expect("Minimum of the log of the flow", year, round(ln_flow.min(), 3), -6.908)
if CHECK_COUNTS:
    print("\nEvery count and share of section 3 of the build plan is reproduced.")

In [ ]:
figure, axes = plt.subplots(1, len(YEARS), figsize=(10, 3.5), sharey=True)
for axis, year in zip(axes, YEARS):
    axis.hist(np.log(clean.loc[clean["year"] == year, TARGET]), bins=60)
    axis.set_title(str(year))
    axis.set_xlabel("Natural log of the flow (flow in thousand USD)")
axes[0].set_ylabel("Rows")
figure.suptitle("Trade flows of the usable rows (display only, not saved)")
figure.tight_layout()
plt.show()